# DBLP Conference Mining Team 7 Notebook

Objective: Menganalisis popularitas konferensi dan menemukan pola asosiasi antar-konferensi dari publikasi inproceedings DBLP 2015-2025 dengan Apriori dan FP-Growth, dinilai dari jumlah frequent itemset, jumlah association rule, dan waktu eksekusi.

# Sinkronisasi versi modules dan libraries

In [ ]:
import sys
print(sys.version)
%pip install -q pandas==2.3.3 numpy==2.4.4 matplotlib==3.10.9 lxml==6.1.3

# Import modules dan libraries

In [ ]:
from collections import Counter, defaultdict
from dataclasses import dataclass
from datetime import datetime, timezone
from itertools import combinations
from pathlib import Path
import gzip
import json
import math
import platform
import re
import sqlite3
import statistics
import time
import unicodedata

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from lxml import etree

# Import datasets

## Snapshot DBLP

### Sumber dan tanggal pengambilan
Snapshot DBLP diunduh satu kali di luar notebook dan disimpan di `data/raw/`, notebook tidak pernah mengunduh ulang, sehingga isi datanya tetap dan angka laporan tidak berubah antar-run. Tanggal pengambilan dan ukuran file dibaca dari `data/raw/metadata.json`.

In [ ]:
SNAPSHOT_DATE = "2026-11-16"
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SNAPSHOT_DIR = ROOT / "data" / "raw"
SNAPSHOT_GZ = SNAPSHOT_DIR / "dblp.xml.gz"
DTD = SNAPSHOT_DIR / "dblp.dtd"
OUT = ROOT / "data" / "processed"
RESULTS = ROOT / "results" / "main"
for folder in (OUT, RESULTS):
    folder.mkdir(parents=True, exist_ok=True)

metadata_file = SNAPSHOT_DIR / "metadata.json"
assert metadata_file.exists(), f"Snapshot belum ada di {SNAPSHOT_DIR}, unduh dblp.xml.gz dan dblp.dtd beserta metadata.json lebih dulu"
snapshot = json.loads(metadata_file.read_text(encoding="utf-8"))
assert SNAPSHOT_GZ.stat().st_size == snapshot["size_bytes"], "Ukuran dblp.xml.gz tidak sama dengan metadata.json"
assert DTD.exists(), f"dblp.dtd tidak ada di {SNAPSHOT_DIR}"

print("Sumber:", snapshot["source"])
print("Tanggal pengambilan:", snapshot["retrieved_at"])
print("Data sampai:", datetime.strptime(SNAPSHOT_DATE, "%Y-%m-%d").strftime("%d/%m/%Y"))
print("Terakhir diubah di server DBLP:", snapshot["last_modified"])
print("Ukuran gzip (GB):", round(snapshot["size_bytes"] / 1e9, 2))

### Parameter eksperimen
Periode pengamatan sesuai soal, empat nilai minimum support untuk Task 4-6, minimum confidence 50%, support untuk analisis temporal, dan jumlah pengulangan pengukuran waktu mining (diambil mediannya). `VENUE_ALIASES` menyatukan NIPS (nama 2015-2017) dengan NeurIPS (nama sejak 2018) karena keduanya konferensi yang sama.

In [ ]:
YEAR_MIN, YEAR_MAX = 2015, 2025
SUPPORTS = [0.001, 0.005, 0.01, 0.02]
MIN_CONFIDENCE = 0.50
TEMPORAL_SUPPORT = 0.005
RUNTIME_REPEATS = 3
VENUE_ALIASES = {"NIPS": "NeurIPS"}

print("Periode:", YEAR_MIN, "-", YEAR_MAX)
print("Minimum support:", SUPPORTS)
print("Minimum confidence:", MIN_CONFIDENCE)
print("Minimum support temporal:", TEMPORAL_SUPPORT)
print("Pengulangan pengukuran waktu:", RUNTIME_REPEATS)

## Ekstraksi XML

### Fungsi pembaca teks
Fungsi untuk membaca teks elemen XML apa adanya dan untuk membersihkan teks (Unicode NFKC dan spasi). Pembacaan mentah dipisah dari pembersihan agar setiap keputusan preprocessing terlihat di Task 1.

In [ ]:
RECORD_TAGS = ("article", "inproceedings", "proceedings", "book", "incollection", "phdthesis", "mastersthesis", "www", "person", "data")
NON_ALNUM = re.compile(r"[\W_]+")


def clean_text(value):
    return re.sub(r"\s+", " ", unicodedata.normalize("NFKC", value or "")).strip()


def raw_text(element, tag):
    child = element.find(tag)
    return "".join(child.itertext()) if child is not None else None


def normalize_author(value):
    return clean_text(value)


def normalize_venue(value):
    name = clean_text(value)
    return VENUE_ALIASES.get(name, name)


def normalize_title(value):
    return NON_ALNUM.sub("", (value or "").casefold())


print("Tag record yang dihitung:", ", ".join(RECORD_TAGS))

### Tabel mentah
Membuat database SQLite berisi dua tabel mentah: `raw_inproceedings` (satu baris per record) dan `raw_authors` (satu baris per author pada record). Ekstraksi dilewati jika database dan ringkasannya sudah ada.

In [ ]:
DB = OUT / "dblp_inproceedings.sqlite"
TYPES_JSON = RESULTS / "record_types.json"
PROFILE_JSON = RESULTS / "inproceedings_profile.json"

extraction_done = DB.exists() and TYPES_JSON.exists() and PROFILE_JSON.exists()
assert extraction_done or not DB.exists(), f"Ekstraksi sebelumnya tidak selesai, ganti nama {DB.name} secara manual agar ekstraksi diulang"
con = sqlite3.connect(DB)
if not extraction_done:
    con.executescript("""
        CREATE TABLE raw_inproceedings (dblp_key TEXT, title TEXT, year TEXT, booktitle TEXT);
        CREATE TABLE raw_authors (dblp_key TEXT, author TEXT);
    """)
print("Ekstraksi sudah selesai sebelumnya" if extraction_done else "Tabel mentah dibuat")

### Membaca XML
Membaca `dblp.xml.gz` satu kali secara bertahap. Setiap record dihitung per jenisnya, dan hanya record `inproceedings` yang disimpan ke tabel mentah tanpa diubah.

In [ ]:
if extraction_done:
    print("Ekstraksi dilewati, database sudah ada:", DB)
else:
    type_counts, field_counts = Counter(), Counter()
    sample_xml = None
    pending_pub, pending_auth = [], []

    def flush():
        con.executemany("INSERT INTO raw_inproceedings VALUES (?,?,?,?)", pending_pub)
        con.executemany("INSERT INTO raw_authors VALUES (?,?)", pending_auth)
        con.commit()
        pending_pub.clear()
        pending_auth.clear()

    parser = etree.XMLPullParser(events=("end",), tag=RECORD_TAGS, load_dtd=True, resolve_entities=True,
                                 no_network=True, huge_tree=True, base_url=str(SNAPSHOT_DIR / "dblp.xml"))
    with gzip.open(SNAPSHOT_GZ, "rb") as stream:
        while chunk := stream.read(1 << 24):
            parser.feed(chunk)
            for _, el in parser.read_events():
                type_counts[el.tag] += 1
                if el.tag == "inproceedings":
                    key, year = el.get("key"), raw_text(el, "year")
                    field_counts.update(set(el.attrib) | {child.tag for child in el})
                    pending_pub.append((key, raw_text(el, "title"), year, raw_text(el, "booktitle")))
                    pending_auth.extend((key, "".join(a.itertext())) for a in el.findall("author"))
                    if sample_xml is None and year and year.strip().isdigit() and YEAR_MIN <= int(year) <= YEAR_MAX:
                        sample_xml = etree.tostring(el, encoding="unicode", with_tail=False)
                el.clear()
                while el.getprevious() is not None:
                    del el.getparent()[0]
            if len(pending_pub) >= 20000:
                flush()
    flush()
    TYPES_JSON.write_text(json.dumps(dict(type_counts.most_common()), indent=2), encoding="utf-8")
    PROFILE_JSON.write_text(json.dumps({"fields": dict(field_counts), "sample_xml": sample_xml}, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Ekstraksi selesai:", dict(type_counts.most_common()))

### Indeks tabel mentah
Membuat indeks pada `dblp_key` agar pencarian author per paper cepat pada langkah preprocessing.

In [ ]:
con.execute("CREATE INDEX IF NOT EXISTS idx_raw_pub_key ON raw_inproceedings (dblp_key)")
con.execute("CREATE INDEX IF NOT EXISTS idx_raw_auth_key ON raw_authors (dblp_key)")
con.commit()
pd.Series({
    "raw_inproceedings (record)": con.execute("SELECT COUNT(*) FROM raw_inproceedings").fetchone()[0],
    "raw_authors (pasangan paper-author)": con.execute("SELECT COUNT(*) FROM raw_authors").fetchone()[0],
}).to_frame("baris")

### Fungsi SQL dan log preprocessing
Mendaftarkan fungsi pembersihan teks ke SQLite supaya langkah preprocessing bisa ditulis sebagai query, dan menyiapkan log jumlah record sebelum dan sesudah setiap langkah.

In [ ]:
con.create_function("clean_text", 1, clean_text, deterministic=True)
con.create_function("normalize_venue", 1, normalize_venue, deterministic=True)
con.create_function("normalize_title", 1, normalize_title, deterministic=True)
preprocessing_log = []


def count_rows(table):
    return con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]


def log_step(step, before, after, note):
    preprocessing_log[:] = [row for row in preprocessing_log if row["langkah"] != step]
    preprocessing_log.append({"langkah": step, "sebelum": before, "sesudah": after, "dibuang": before - after, "keterangan": note})
    print(f"{step}: {before:,} -> {after:,} record (dibuang {before - after:,})")

# Task 1 - Data Understanding and Preprocessing

## 1. Eksplorasi dataset
Menjelaskan struktur dataset, atribut, jumlah record, dan jenis publikasi yang tersedia.

### Struktur dataset
Dump berupa satu file XML dengan elemen root `<dblp>` yang berisi record satu per satu. Setiap record punya atribut `key` (identitas unik yang memuat kode venue, misalnya `conf/nips/...`) dan `mdate` (tanggal record diubah), lalu elemen isi seperti pada contoh record `inproceedings` berikut.

In [ ]:
profile = json.loads(PROFILE_JSON.read_text(encoding="utf-8"))
record_types = pd.Series(json.loads(TYPES_JSON.read_text(encoding="utf-8")), name="records").sort_values(ascending=False)
n_inproceedings = int(record_types["inproceedings"])
print(profile["sample_xml"])

### Atribut dataset
Atribut pada record `inproceedings` beserta jumlah record yang mengisinya. Atribut yang dipakai analisis:
- key: identitas unik record, memuat kode venue.
- author: penulis paper, nol atau lebih per record.
- title: judul paper.
- year: tahun acara konferensi.
- booktitle: nama prosiding, dipakai sebagai nama venue.

Atribut lain (`pages`, `ee`, `url`, `crossref`, dan seterusnya) tidak relevan untuk popularitas dan pola asosiasi.

In [ ]:
USED_ATTRIBUTES = ["key", "author", "title", "year", "booktitle"]
attributes = pd.Series(profile["fields"], name="records_with_value").to_frame()
attributes["missing"] = n_inproceedings - attributes["records_with_value"]
attributes["missing_%"] = (attributes["missing"] / n_inproceedings * 100).round(3)
attributes["used"] = attributes.index.isin(USED_ATTRIBUTES)
attributes = attributes.sort_values(["used", "records_with_value"], ascending=False)
attributes

### Jumlah record
Jumlah record pada dump, berapa banyak di antaranya yang berjenis `inproceedings`, dan jumlah pasangan paper-author di dalamnya.

In [ ]:
total_records = int(record_types.sum())
pd.Series({
    "record pada dump (semua jenis)": total_records,
    "record inproceedings": n_inproceedings,
    "porsi inproceedings (%)": round(n_inproceedings / total_records * 100, 2),
    "pasangan paper-author pada inproceedings": count_rows("raw_authors"),
}).to_frame("nilai")

### Jenis publikasi
Jumlah record per jenis publikasi pada seluruh dump. Jenis yang ada:
- article: paper jurnal.
- www: halaman profil author.
- inproceedings: paper konferensi dan workshop.
- phdthesis dan mastersthesis: disertasi doktor dan tesis magister.
- incollection: bab dalam buku.
- proceedings: data prosiding itu sendiri, bukan paper di dalamnya.
- book: buku.
- data: dataset penelitian.

In [ ]:
record_type_table = record_types.to_frame()
record_type_table["share_%"] = (record_type_table["records"] / total_records * 100).round(2)
record_type_table

## 2. Preprocessing
Membersihkan record `inproceedings` menjadi tabel `publications` (satu baris per paper) dan `authorship` (satu baris per pasangan paper dan author). Langkah dijalankan berurutan dan setiap langkah mencatat jumlah record yang dibuang. Tahun difilter lebih awal karena langkah ini yang paling banyak mengurangi data, sehingga langkah sesudahnya lebih ringan.

### Memilih publikasi konferensi dan workshop
Hanya record `inproceedings` yang dipakai karena itu jenis record DBLP untuk paper konferensi dan workshop. Jurnal, buku, dan halaman profil tidak punya venue konferensi sehingga tidak bisa menjadi item. Pemilihan terjadi saat ekstraksi, langkah ini menyalin hasilnya ke tabel kerja `publications`.

In [ ]:
con.executescript("""
DROP TABLE IF EXISTS publications;
DROP TABLE IF EXISTS authorship;
CREATE TABLE publications (dblp_key TEXT, title TEXT, year INTEGER, booktitle TEXT, venue TEXT);
CREATE TABLE authorship (dblp_key TEXT, author TEXT);
INSERT INTO publications (dblp_key, title, year, booktitle)
    SELECT dblp_key, title, TRIM(year), booktitle FROM raw_inproceedings;
CREATE INDEX idx_publications_key ON publications (dblp_key);
""")
log_step("Memilih konferensi dan workshop", total_records, count_rows("publications"), "hanya record inproceedings")

### Konsistensi atribut tahun
`year` dibaca sebagai integer dan hanya tahun 2015-2025 yang dipakai, agar tahun bisa dibandingkan sebagai angka dan sesuai periode pengamatan. Record dengan tahun yang bukan angka atau di luar periode dibuang.

In [ ]:
before = count_rows("publications")
con.execute("DELETE FROM publications WHERE typeof(year) <> 'integer' OR year NOT BETWEEN ? AND ?", (YEAR_MIN, YEAR_MAX))
con.commit()
log_step("Konsistensi tahun", before, count_rows("publications"), f"year integer {YEAR_MIN}-{YEAR_MAX}")
pd.read_sql_query("SELECT MIN(year) AS year_min, MAX(year) AS year_max, COUNT(DISTINCT year) AS n_years FROM publications", con)

### Menangani missing values
Record tanpa key, title, booktitle (venue), atau author dibuang karena tanpa salah satunya record tidak bisa menjadi transaksi, dan jumlahnya terlalu kecil untuk memengaruhi hasil.

In [ ]:
before = count_rows("publications")
without_text = con.execute("""
    DELETE FROM publications WHERE dblp_key IS NULL OR title IS NULL OR TRIM(title) = ''
        OR booktitle IS NULL OR TRIM(booktitle) = ''
""").rowcount
without_author = con.execute("""
    DELETE FROM publications WHERE NOT EXISTS (
        SELECT 1 FROM raw_authors a WHERE a.dblp_key = publications.dblp_key AND TRIM(a.author) <> '')
""").rowcount
con.commit()
print(f"Dibuang karena key, title, atau booktitle kosong: {without_text:,}")
print(f"Dibuang karena tidak punya author: {without_author:,}")
log_step("Menangani missing values", before, count_rows("publications"), "tanpa key, title, booktitle, atau author")

### Normalisasi nama author
Normalisasi Unicode (NFKC) dan spasi pada nama author, lalu setiap pasangan paper-author disimpan sekali. Suffix homonim DBLP seperti "Wei Wang 0001" dipertahankan karena itu penanda orang yang berbeda, kalau dihapus banyak peneliti berbeda tergabung menjadi satu author.

In [ ]:
before = count_rows("publications")
con.execute("DELETE FROM authorship")
con.execute("""
    INSERT INTO authorship (dblp_key, author)
    SELECT DISTINCT dblp_key, author FROM (
        SELECT p.dblp_key AS dblp_key, clean_text(a.author) AS author
        FROM publications p JOIN raw_authors a ON a.dblp_key = p.dblp_key
    ) WHERE author <> ''
""")
without_author = con.execute("""
    DELETE FROM publications WHERE NOT EXISTS (SELECT 1 FROM authorship a WHERE a.dblp_key = publications.dblp_key)
""").rowcount
con.execute("CREATE INDEX IF NOT EXISTS idx_authorship_key ON authorship (dblp_key)")
con.execute("CREATE INDEX IF NOT EXISTS idx_authorship_author ON authorship (author)")
con.commit()
names_raw = con.execute("""
    SELECT COUNT(DISTINCT a.author) FROM raw_authors a
    WHERE EXISTS (SELECT 1 FROM publications p WHERE p.dblp_key = a.dblp_key)
""").fetchone()[0]
names_clean = con.execute("SELECT COUNT(DISTINCT author) FROM authorship").fetchone()[0]
homonyms = pd.read_sql_query("SELECT DISTINCT author FROM authorship WHERE author GLOB '* [0-9][0-9][0-9][0-9]'", con)
print(f"Nama author: {names_raw:,} mentah -> {names_clean:,} setelah normalisasi")
print(f"Author dengan suffix homonim: {len(homonyms):,}, paper tanpa author valid dibuang: {without_author:,}")
log_step("Normalisasi nama author", before, count_rows("publications"), f"{names_raw:,} nama mentah -> {names_clean:,} author")
homonyms.head(5)

### Normalisasi nama conference/venue
Nama venue diambil dari `booktitle` setelah Unicode dan spasi dirapikan, lalu `VENUE_ALIASES` diterapkan (NIPS menjadi NeurIPS). Tabel di bawah mengaudit nama lain yang tampak sinonim, misalnya variasi nomor jilid dalam tanda kurung seperti "ECCV (1)" dan "ECCV (2)".

In [ ]:
before = count_rows("publications")
con.execute("UPDATE publications SET venue = normalize_venue(booktitle)")
empty_venue = con.execute("DELETE FROM publications WHERE venue = ''").rowcount
con.commit()
venues = pd.read_sql_query("SELECT venue, COUNT(*) AS publications FROM publications GROUP BY venue", con)
venues["base_name"] = venues["venue"].str.replace(r"\s*\(.*\)\s*$", "", regex=True)
variants = venues.groupby("base_name").agg(variants=("venue", "nunique"), publications=("publications", "sum"))
booktitles = con.execute("SELECT COUNT(DISTINCT booktitle) FROM publications").fetchone()[0]
aliased = con.execute("SELECT COUNT(*) FROM publications WHERE booktitle <> venue AND booktitle IN (%s)" % ",".join("?" * len(VENUE_ALIASES)), list(VENUE_ALIASES)).fetchone()[0]
print(f"Paper yang namanya diganti lewat alias {VENUE_ALIASES}: {aliased:,}")
print(f"Booktitle mentah: {booktitles:,} -> venue: {len(venues):,}, nama dasar unik setelah tanda kurung dibuang: {len(variants):,}")
log_step("Normalisasi nama venue", before, count_rows("publications"), f"{booktitles:,} booktitle -> {len(venues):,} venue")
variants.sort_values("variants", ascending=False).head(10)

### Menghilangkan data duplikat
Record dianggap duplikat jika `key`-nya sama, atau judul (huruf kecil tanpa tanda baca), kumpulan author, venue, dan tahunnya sama. Hanya record pertama yang disimpan, karena satu paper tidak boleh dihitung dua kali.

In [ ]:
before = count_rows("publications")
same_key = con.execute("DELETE FROM publications WHERE rowid NOT IN (SELECT MIN(rowid) FROM publications GROUP BY dblp_key)").rowcount
con.execute("DROP TABLE IF EXISTS temp.author_signature")
con.execute("""
    CREATE TEMP TABLE author_signature AS
    SELECT dblp_key, group_concat(author, '|') AS authors
    FROM (SELECT dblp_key, author FROM authorship ORDER BY dblp_key, author)
    GROUP BY dblp_key
""")
same_content = con.execute("""
    DELETE FROM publications WHERE rowid IN (
        SELECT rowid FROM (
            SELECT p.rowid AS rowid, ROW_NUMBER() OVER (
                PARTITION BY normalize_title(p.title), s.authors, p.venue, p.year ORDER BY p.rowid) AS rn
            FROM publications p JOIN temp.author_signature s ON s.dblp_key = p.dblp_key
        ) WHERE rn > 1
    )
""").rowcount
orphans = con.execute("""
    DELETE FROM authorship WHERE NOT EXISTS (SELECT 1 FROM publications p WHERE p.dblp_key = authorship.dblp_key)
""").rowcount
con.commit()
print(f"Dibuang karena key sama: {same_key:,}")
print(f"Dibuang karena judul, author, venue, dan tahun sama: {same_content:,}")
print(f"Baris authorship milik paper yang dibuang: {orphans:,}")
log_step("Menghilangkan data duplikat", before, count_rows("publications"), "key sama atau judul, author, venue, tahun sama")

## 3. Ringkasan tahapan preprocessing
Jumlah record yang tersisa setelah setiap tahapan preprocessing beserta jumlah yang dibuang.

In [ ]:
pd.DataFrame(preprocessing_log)

## 4. Statistik dataset setelah preprocessing

### Total publikasi
Jumlah paper yang tersisa setelah preprocessing.

In [ ]:
n_publications = count_rows("publications")
print(f"Total publikasi: {n_publications:,}")

### Total author unik
Jumlah nama author yang berbeda setelah normalisasi.

In [ ]:
n_authors = con.execute("SELECT COUNT(DISTINCT author) FROM authorship").fetchone()[0]
print(f"Total author unik: {n_authors:,}")

### Total conference/venue unik
Jumlah venue yang berbeda setelah normalisasi.

In [ ]:
n_venues = con.execute("SELECT COUNT(DISTINCT venue) FROM publications").fetchone()[0]
stats = pd.DataFrame({"publications": [n_publications], "unique_authors": [n_authors], "unique_venues": [n_venues]})
stats.to_csv(RESULTS / "task1_stats.csv", index=False)
print(f"Total conference/venue unik: {n_venues:,}")
stats

### Distribusi publikasi berdasarkan tahun
Jumlah publikasi pada setiap tahun acara 2015-2025.

In [ ]:
by_year = pd.read_sql_query("SELECT year, COUNT(*) AS publications FROM publications GROUP BY year ORDER BY year", con)
by_year.to_csv(RESULTS / "task1_by_year.csv", index=False)
display(by_year)
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(by_year.year, by_year.publications)
ax.set(xlabel="Tahun acara", ylabel="Jumlah publikasi", title="Publikasi inproceedings DBLP, 2015-2025")
plt.tight_layout()
plt.show()

### Distribusi publikasi berdasarkan conference
Jumlah publikasi pada setiap venue, ditampilkan 20 venue teratas.

In [ ]:
by_venue = pd.read_sql_query("SELECT venue, COUNT(*) AS publications FROM publications GROUP BY venue ORDER BY publications DESC, venue", con)
by_venue.to_csv(RESULTS / "task1_by_venue.csv", index=False)
display(by_venue.head(20))
fig, ax = plt.subplots(figsize=(10, 6))
by_venue.head(20).iloc[::-1].plot.barh(x="venue", y="publications", ax=ax, legend=False, title="20 venue dengan publikasi terbanyak")
plt.tight_layout()
plt.show()

# Task 2 - Conference Popularity Analysis
Popularitas konferensi diukur dari aktivitas publikasi, memakai nama venue hasil normalisasi di Task 1 (NIPS sudah menjadi NeurIPS).

## 1. 10 konferensi dengan jumlah publikasi terbanyak
Jumlah publikasi dihitung per venue.

In [ ]:
publications_by_venue = pd.read_sql_query("""
SELECT venue, COUNT(*) AS publications FROM publications
GROUP BY venue ORDER BY publications DESC, venue
""", con)
top_pubs = publications_by_venue.head(10)
top_pubs.to_csv(RESULTS / "task2_top_publications.csv", index=False)
display(top_pubs)
top_pubs.iloc[::-1].plot.barh(x="venue", y="publications", legend=False, title="Top 10 publikasi", figsize=(7, 5))
plt.tight_layout()
plt.show()

## 2. 10 konferensi dengan jumlah author unik terbanyak
Author unik dihitung per venue setelah join publikasi dan authorship, sehingga author dengan beberapa paper di venue yang sama dihitung sekali.

In [ ]:
authors_by_venue = pd.read_sql_query("""
SELECT p.venue AS venue, COUNT(DISTINCT a.author) AS unique_authors
FROM publications p JOIN authorship a USING (dblp_key)
GROUP BY p.venue ORDER BY unique_authors DESC, venue
""", con)
top_authors = authors_by_venue.head(10)
top_authors.to_csv(RESULTS / "task2_top_authors.csv", index=False)
display(top_authors)
top_authors.iloc[::-1].plot.barh(x="venue", y="unique_authors", legend=False, title="Top 10 author unik", figsize=(7, 5))
plt.tight_layout()
plt.show()

## 3. Tren jumlah publikasi berdasarkan tahun 2015-2025
Jumlah publikasi per tahun untuk venue yang masuk lima besar pada salah satu ranking di atas.

In [ ]:
leaders = tuple(dict.fromkeys(top_pubs.venue.head(5).tolist() + top_authors.venue.head(5).tolist()))
trend = pd.read_sql_query("SELECT year, venue, COUNT(*) AS publications FROM publications GROUP BY year, venue", con)
trend.to_csv(RESULTS / "task2_trend_all_venues.csv", index=False)
trend_pivot = trend[trend.venue.isin(leaders)].pivot(index="year", columns="venue", values="publications").fillna(0)
trend_pivot.plot(figsize=(11, 5), marker="o", title="Tren venue per tahun")
plt.ylabel("Publikasi")
plt.tight_layout()
plt.show()

## 4. Perbandingan ranking publikasi dan author unik
Menggabungkan kedua ranking. `rank_difference` bernilai positif jika venue lebih tinggi pada ranking author unik daripada ranking publikasi.

In [ ]:
ranking = publications_by_venue.merge(authors_by_venue, on="venue", how="outer")
ranking["rank_publications"] = ranking["publications"].rank(ascending=False, method="min").astype(int)
ranking["rank_unique_authors"] = ranking["unique_authors"].rank(ascending=False, method="min").astype(int)
ranking["rank_difference"] = ranking["rank_publications"] - ranking["rank_unique_authors"]
in_top10 = ranking[(ranking["rank_publications"] <= 10) | (ranking["rank_unique_authors"] <= 10)]
overlap = len(set(top_pubs.venue) & set(top_authors.venue))
print(f"Venue yang masuk 10 besar pada kedua ranking: {overlap} dari 10")
in_top10.sort_values("rank_publications")

# Task 3 - Transactional Database Construction
Setiap item adalah nama venue. Transaksi disimpan sebagai himpunan sehingga paper berulang di venue yang sama tidak menggandakan item, dan transaksi satu item tetap dihitung pada penyebut support.

## 1. Membentuk transactional database

### Author-based transaction
Setiap transaksi adalah satu author, itemnya adalah semua venue tempat author itu pernah publikasi selama 2015-2025.

In [ ]:
author_venue = pd.read_sql_query("""
SELECT DISTINCT a.author, p.venue FROM authorship a JOIN publications p USING (dblp_key)
ORDER BY a.author, p.venue
""", con)
author_tx = author_venue.groupby("author", sort=False).venue.agg(lambda s: tuple(sorted(set(s)))).reset_index(name="items")
del author_venue
author_tx.to_pickle(OUT / "author_transactions.pkl")
print("Transaksi author-based:", f"{len(author_tx):,}")

### Author-year-based transaction
Setiap transaksi adalah satu kombinasi author dan tahun, itemnya adalah venue tempat author itu publikasi pada tahun tersebut.

In [ ]:
author_year_venue = pd.read_sql_query("""
SELECT DISTINCT a.author, p.year, p.venue FROM authorship a JOIN publications p USING (dblp_key)
ORDER BY a.author, p.year, p.venue
""", con)
author_year_tx = author_year_venue.groupby(["author", "year"], sort=False).venue.agg(lambda s: tuple(sorted(set(s)))).reset_index(name="items")
del author_year_venue
author_year_tx.to_pickle(OUT / "author_year_transactions.pkl")
print("Transaksi author-year-based:", f"{len(author_year_tx):,}")

## 2. Contoh transaksi
Sepuluh transaksi yang diambil acak (`random_state=42`) dari masing-masing representasi.

### Author-based transaction

In [ ]:
author_tx.sample(10, random_state=42)

### Author-year-based transaction

In [ ]:
author_year_tx.sample(10, random_state=42)

## 3. Jumlah transaksi
Jumlah transaksi pada masing-masing transactional database.

In [ ]:
pd.Series({"Author-based": len(author_tx), "Author-year-based": len(author_year_tx)}).to_frame("transaksi")

## 4. Item unik dan distribusi item per transaksi

### Jumlah item unik
Jumlah venue berbeda yang muncul sebagai item pada masing-masing representasi.

In [ ]:
pd.Series({
    "Author-based": len({x for row in author_tx["items"] for x in row}),
    "Author-year-based": len({x for row in author_year_tx["items"] for x in row}),
}).to_frame("item unik")

### Distribusi jumlah item per transaksi
Statistik jumlah item per transaksi dan histogramnya, dengan transaksi berisi 20 item atau lebih digabung pada satu batang.

In [ ]:
sizes = {"Author-based": author_tx["items"].map(len), "Author-year-based": author_year_tx["items"].map(len)}
display(pd.DataFrame({name: s.describe() for name, s in sizes.items()}))
display(pd.DataFrame({name: (s == 1).mean() for name, s in sizes.items()}, index=["porsi transaksi satu item"]))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (name, s) in zip(axes, sizes.items()):
    s.clip(upper=20).hist(ax=ax, bins=20)
    ax.set(xlabel="Item per transaksi (20 = 20+)", ylabel="Jumlah transaksi", title=name)
plt.tight_layout()
plt.show()

# Task 4 - Frequent Pattern Mining
Apriori dan FP-Growth diimplementasikan sendiri (tanpa library mining) dan dijalankan pada transaksi Author-Based dengan empat nilai minimum support. Keduanya memakai input dan format hasil yang sama, sehingga bisa dibandingkan langsung di Task 6.

## 1. Implementasi Apriori dan FP-Growth

### Transaksi terkode
Mengubah nama venue menjadi nomor item dan menyimpan transaksi yang identik satu kali beserta jumlah kemunculannya. Kedua algoritma jadi membaca jauh lebih sedikit baris, sementara support count tetap tepat.

In [ ]:
@dataclass
class TransactionDB:
    weighted: dict
    n_transactions: int
    items: list

    def min_count(self, min_support):
        return max(1, math.ceil(min_support * self.n_transactions - 1e-9))


def encode_transactions(item_tuples):
    occurrences = item_tuples.value_counts()
    items = sorted({item for transaction in occurrences.index for item in transaction})
    item_id = {item: i for i, item in enumerate(items)}
    weighted = {tuple(sorted(item_id[item] for item in transaction)): int(count)
                for transaction, count in occurrences.items()}
    return TransactionDB(weighted, int(occurrences.sum()), items)


author_db = encode_transactions(author_tx["items"])
print(f"Transaksi: {author_db.n_transactions:,} | transaksi berbeda: {len(author_db.weighted):,} | item: {len(author_db.items):,}")

### Apriori
Apriori bekerja per level. Kandidat k-itemset dibentuk dari frequent (k-1)-itemset yang berbagi k-2 item pertama, kandidat yang punya subset tidak frequent dibuang (sifat Apriori), lalu satu pemindaian data menghitung support kandidat yang tersisa.

In [ ]:
def generate_candidates(frequent, k):
    by_prefix = defaultdict(list)
    for itemset in frequent:
        by_prefix[itemset[:-1]].append(itemset[-1])
    candidates = set()
    for prefix, last_items in by_prefix.items():
        last_items.sort()
        for a, b in combinations(last_items, 2):
            candidate = prefix + (a, b)
            if all(subset in frequent for subset in combinations(candidate, k - 1)):
                candidates.add(candidate)
    return candidates


def count_candidates(transactions, candidates, k):
    counts = dict.fromkeys(candidates, 0)
    for transaction, weight in transactions.items():
        if math.comb(len(transaction), k) <= len(candidates):
            for subset in combinations(transaction, k):
                if subset in counts:
                    counts[subset] += weight
        else:
            items = set(transaction)
            for candidate in candidates:
                if items.issuperset(candidate):
                    counts[candidate] += weight
    return counts


def reduce_transactions(transactions, keep, min_length):
    reduced = Counter()
    for transaction, weight in transactions.items():
        kept = tuple(item for item in transaction if item in keep)
        if len(kept) >= min_length:
            reduced[kept] += weight
    return reduced


def apriori(db, min_support):
    min_count = db.min_count(min_support)
    item_counts = Counter()
    for transaction, weight in db.weighted.items():
        for item in transaction:
            item_counts[item] += weight
    frequent = {(item,): count for item, count in item_counts.items() if count >= min_count}
    result = dict(frequent)
    transactions = db.weighted
    k = 2
    while frequent:
        candidates = generate_candidates(set(frequent), k)
        if not candidates:
            break
        candidate_items = {item for candidate in candidates for item in candidate}
        transactions = reduce_transactions(transactions, candidate_items, k)
        counts = count_candidates(transactions, candidates, k)
        frequent = {itemset: count for itemset, count in counts.items() if count >= min_count}
        result.update(frequent)
        k += 1
    return result


print("Apriori siap, hasil berupa {itemset: support count}")

### FP-Growth
FP-Growth membaca data dua kali (menghitung item, lalu menyusun FP-tree) dan tidak membuat kandidat. Pola dikembangkan langsung dari pohon, dengan conditional FP-tree yang dibangun dari jalur menuju tiap item.

In [ ]:
class Node:
    __slots__ = ("item", "count", "parent", "children", "link")

    def __init__(self, item, parent):
        self.item = item
        self.count = 0
        self.parent = parent
        self.children = {}
        self.link = None


def build_tree(transactions, min_count):
    transactions = list(transactions)
    item_counts = Counter()
    for items, weight in transactions:
        for item in items:
            item_counts[item] += weight
    item_counts = {item: count for item, count in item_counts.items() if count >= min_count}
    root = Node(None, None)
    header = {}
    for items, weight in transactions:
        ordered = sorted((item for item in items if item in item_counts), key=lambda item: (-item_counts[item], item))
        node = root
        for item in ordered:
            child = node.children.get(item)
            if child is None:
                child = Node(item, node)
                node.children[item] = child
                child.link = header.get(item)
                header[item] = child
            child.count += weight
            node = child
    return header, item_counts


def mine_tree(header, item_counts, suffix, min_count, result):
    for item in sorted(item_counts, key=lambda item: (item_counts[item], item)):
        pattern = suffix + (item,)
        result[tuple(sorted(pattern))] = item_counts[item]
        conditional_base = []
        node = header[item]
        while node is not None:
            path = []
            parent = node.parent
            while parent.item is not None:
                path.append(parent.item)
                parent = parent.parent
            if path:
                conditional_base.append((path, node.count))
            node = node.link
        if conditional_base:
            cond_header, cond_counts = build_tree(conditional_base, min_count)
            if cond_counts:
                mine_tree(cond_header, cond_counts, pattern, min_count, result)


def fpgrowth(db, min_support):
    min_count = db.min_count(min_support)
    header, item_counts = build_tree(db.weighted.items(), min_count)
    result = {}
    mine_tree(header, item_counts, (), min_count, result)
    return result


print("FP-Growth siap, hasil berupa {itemset: support count}")

### Fungsi eksperimen
Mengubah hasil algoritma menjadi tabel (itemset, length, support_count, support) dan menjalankan satu algoritma pada satu minimum support. Waktu yang dicatat adalah median dari beberapa kali jalan dan hanya mencakup waktu mining. `run_experiment` menjalankan kedua algoritma pada satu support dan memeriksa hasilnya sama.

In [ ]:
mining = {}


def itemsets_to_frame(itemsets, db):
    frame = pd.DataFrame({
        "itemset": [tuple(sorted(db.items[item] for item in itemset)) for itemset in itemsets],
        "length": [len(itemset) for itemset in itemsets],
        "support_count": list(itemsets.values()),
    })
    frame["support"] = frame["support_count"] / db.n_transactions
    return frame.sort_values(["support_count", "itemset"], ascending=[False, True]).reset_index(drop=True)


def mine(db, algorithm, min_support, repeats=1):
    algorithms = {"Apriori": apriori, "FP-Growth": fpgrowth}
    runtimes = []
    for _ in range(repeats):
        start = time.perf_counter()
        raw_itemsets = algorithms[algorithm](db, min_support)
        runtimes.append(time.perf_counter() - start)
    return itemsets_to_frame(raw_itemsets, db), statistics.median(runtimes)


def join_items(frame, columns):
    return frame.assign(**{column: frame[column].str.join(" | ") for column in columns})


def run_experiment(support):
    for algorithm in ("Apriori", "FP-Growth"):
        found, seconds = mine(author_db, algorithm, support, RUNTIME_REPEATS)
        mining[(algorithm, support)] = {"itemsets": found, "seconds": seconds}
        join_items(found, ["itemset"]).to_csv(RESULTS / f"task4_{algorithm.lower().replace('-', '')}_{support}.csv", index=False)
        print(f"{algorithm} support={support:.1%}: {len(found):,} itemset, {seconds:.2f} detik (median {RUNTIME_REPEATS} kali)")
    a, f = mining[("Apriori", support)]["itemsets"], mining[("FP-Growth", support)]["itemsets"]
    print("Itemset dan support count Apriori dan FP-Growth sama:", dict(zip(a.itemset, a.support_count)) == dict(zip(f.itemset, f.support_count)))


def mining_summary_table():
    rows = []
    for (algorithm, support), result in mining.items():
        found = result["itemsets"]
        lengths = found["length"].value_counts()
        rows.append({"algorithm": algorithm, "min_support": support, "seconds": result["seconds"],
                     "itemsets": len(found), "max_length": int(found["length"].max()) if len(found) else 0,
                     **{f"{k}-itemsets": int(lengths.get(k, 0)) for k in (1, 2, 3)}})
    return pd.DataFrame(rows).sort_values(["min_support", "algorithm"]).reset_index(drop=True)

## 2. Frequent itemsets untuk setiap minimum support
Setiap sel menjalankan Apriori dan FP-Growth pada satu minimum support.

### Minimum support 0,1%

In [ ]:
run_experiment(0.001)

### Minimum support 0,5%

In [ ]:
run_experiment(0.005)

### Minimum support 1%

In [ ]:
run_experiment(0.01)

### Minimum support 2%

In [ ]:
run_experiment(0.02)

## 3. Frequent 1-itemsets, 2-itemsets, dan 3-itemsets
Jumlah frequent itemset menurut ukurannya pada setiap minimum support.

In [ ]:
mining_summary = mining_summary_table()
mining_summary.to_csv(RESULTS / "task4_summary.csv", index=False)
mining_summary[["algorithm", "min_support", "itemsets", "1-itemsets", "2-itemsets", "3-itemsets", "max_length"]]

## 4. 10 frequent itemsets dengan support tertinggi
Itemset diambil dari hasil FP-Growth, yang identik dengan hasil Apriori. Kolom `support_count` adalah jumlah author yang memuat itemset tersebut.

In [ ]:
def top_itemsets(support, n=10, min_length=1):
    found = mining[("FP-Growth", support)]["itemsets"]
    return join_items(found[found["length"] >= min_length].head(n).reset_index(drop=True), ["itemset"])

### Minimum support 0,1%

In [ ]:
top_itemsets(0.001)

### Minimum support 0,5%

In [ ]:
top_itemsets(0.005)

### Minimum support 1%

In [ ]:
top_itemsets(0.01)

### Minimum support 2%

In [ ]:
top_itemsets(0.02)

## 5. Pengaruh minimum support terhadap jumlah frequent itemsets
Jumlah frequent itemset pada setiap minimum support, dengan skala log pada sumbu jumlah.

In [ ]:
influence = mining_summary[mining_summary["algorithm"] == "FP-Growth"].set_index("min_support")[["itemsets", "1-itemsets", "2-itemsets", "3-itemsets", "max_length"]]
display(influence)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(influence.index * 100, influence["itemsets"], marker="o")
ax.set(xlabel="Minimum support (%)", ylabel="Jumlah frequent itemset", yscale="log")
ax.grid(alpha=.2)
plt.tight_layout()
plt.show()

## 6. Kombinasi conference yang paling sering muncul bersama
Sepuluh itemset berisi dua venue atau lebih dengan support tertinggi, diambil dari hasil minimum support 0,1% karena mencakup semua itemset yang lolos pada support yang lebih tinggi.

In [ ]:
top_itemsets(0.001, min_length=2)

# Task 5 - Association Rule Mining
Rule dibangkitkan dari frequent itemsets FP-Growth dengan minimum confidence 50%. Lift adalah confidence dibagi support consequent, lift di atas 1 berarti kedua sisi muncul bersama lebih sering daripada perkiraan jika keduanya independen.

## 1. Membangkitkan association rules

### Fungsi pembangkit rule
Untuk setiap frequent itemset I dan setiap pembagian menjadi antecedent A dan consequent B, support = count(I) / N, confidence = count(I) / count(A), dan lift = confidence / support(B). Semua subset dari itemset yang frequent juga frequent, sehingga count(A) dan count(B) selalu tersedia.

In [ ]:
RULE_COLUMNS = ["antecedent", "consequent", "support", "confidence", "lift", "support_count", "antecedent_support", "consequent_support"]


def association_rules(itemsets, n_transactions, min_confidence=MIN_CONFIDENCE):
    counts = dict(zip(itemsets["itemset"], itemsets["support_count"]))
    rows = []
    for itemset, count in counts.items():
        if len(itemset) < 2:
            continue
        for size in range(1, len(itemset)):
            for antecedent in combinations(itemset, size):
                confidence = count / counts[antecedent]
                if confidence < min_confidence:
                    continue
                consequent = tuple(item for item in itemset if item not in antecedent)
                consequent_support = counts[consequent] / n_transactions
                rows.append((antecedent, consequent, count / n_transactions, confidence, confidence / consequent_support,
                             count, counts[antecedent] / n_transactions, consequent_support))
    rules = pd.DataFrame(rows, columns=RULE_COLUMNS)
    rules = rules.astype({column: float for column in RULE_COLUMNS[2:]}).astype({"support_count": int})
    return rules.sort_values(["lift", "confidence", "support"], ascending=False).reset_index(drop=True)


def format_rules(rules):
    return pd.DataFrame({
        "rule": rules["antecedent"].str.join(", ") + "  ->  " + rules["consequent"].str.join(", "),
        "support": rules["support"].round(5),
        "confidence": rules["confidence"].round(3),
        "lift": rules["lift"].round(2),
        "support_count": rules["support_count"],
    }).reset_index(drop=True)


def show_top_rules(support, n=10):
    rules = rules_by_support[support]
    if rules.empty:
        return f"Tidak ada rule pada minimum support {support:.1%}"
    return format_rules(rules.head(n))

### Rule untuk setiap minimum support
Membangkitkan rule dari hasil FP-Growth pada keempat minimum support dan menyimpannya ke `results/main/`.

In [ ]:
rules_by_support = {}
for support in SUPPORTS:
    rules = association_rules(mining[("FP-Growth", support)]["itemsets"], author_db.n_transactions)
    rules_by_support[support] = rules
    join_items(rules, ["antecedent", "consequent"]).to_csv(RESULTS / f"task5_rules_{support}.csv", index=False)
    print(f"support={support:.1%}: {len(rules):,} rules")

## 2. 10 association rules dengan lift tertinggi
Rule diurutkan menurut lift, lalu confidence dan support, lengkap dengan nilai support, confidence, dan lift.

### Minimum support 0,1%

In [ ]:
show_top_rules(0.001)

### Minimum support 0,5%

In [ ]:
show_top_rules(0.005)

### Minimum support 1%

In [ ]:
show_top_rules(0.01)

### Minimum support 2%

In [ ]:
show_top_rules(0.02)

## 3. Support tinggi dan lift
Memeriksa apakah antecedent dengan support tinggi selalu menghasilkan lift tinggi, pada minimum support 0,5%. Korelasi Spearman antara support dan lift ditampilkan bersama sepuluh rule dengan antecedent support tertinggi.

In [ ]:
chosen = rules_by_support[0.005]
if chosen.empty:
    print("Tidak ada rule pada minimum support 0,5%")
else:
    print("Korelasi Spearman support vs lift:", round(chosen["support"].corr(chosen["lift"], method="spearman"), 3))
    display(join_items(chosen.sort_values("antecedent_support", ascending=False).head(10), ["antecedent", "consequent"])
            [["antecedent", "consequent", "antecedent_support", "support", "confidence", "lift"]].reset_index(drop=True))

# Task 6 - Algorithm Performance Comparison
Apriori dan FP-Growth dinilai pada transaksi dan minimum support yang sama. Waktu hanya waktu mining, tidak termasuk pembuatan transaksi terkode, dan dipengaruhi kondisi mesin saat dijalankan.

## 1. Jumlah frequent itemsets
Jumlah frequent itemset per algoritma pada setiap minimum support.

In [ ]:
comparison = mining_summary_table()
comparison.pivot(index="min_support", columns="algorithm", values="itemsets")

## 2. Jumlah association rules
Jumlah rule yang dihasilkan dari frequent itemsets masing-masing algoritma pada minimum confidence 50%.

In [ ]:
comparison["rules"] = comparison.apply(
    lambda row: len(association_rules(mining[(row["algorithm"], row["min_support"])]["itemsets"], author_db.n_transactions)), axis=1
)
comparison.pivot(index="min_support", columns="algorithm", values="rules")

## 3. Waktu eksekusi

### Tabel waktu
Waktu mining dalam detik per algoritma pada setiap minimum support.

In [ ]:
comparison.pivot(index="min_support", columns="algorithm", values="seconds").round(2)

### Grafik performa
Jumlah frequent itemset dan waktu mining terhadap minimum support untuk kedua algoritma.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for alg, part in comparison.groupby("algorithm"):
    axes[0].plot(part.min_support * 100, part.itemsets, marker="o", label=alg)
    axes[1].plot(part.min_support * 100, part.seconds, marker="o", label=alg)
axes[0].set(xlabel="Minimum support (%)", ylabel="Jumlah itemset")
axes[1].set(xlabel="Minimum support (%)", ylabel="Detik")
for ax in axes:
    ax.legend()
    ax.grid(alpha=.2)
plt.tight_layout()
plt.show()

## 4. Kesamaan frequent itemsets
Memastikan Apriori dan FP-Growth menghasilkan itemset dan support count yang identik pada setiap minimum support.

In [ ]:
checks = []
for support in SUPPORTS:
    a, f = mining[("Apriori", support)]["itemsets"], mining[("FP-Growth", support)]["itemsets"]
    checks.append({"min_support": support, "itemsets": len(a),
                   "same_itemsets_and_support": dict(zip(a.itemset, a.support_count)) == dict(zip(f.itemset, f.support_count))})
assert all(x["same_itemsets_and_support"] for x in checks), "Apriori dan FP-Growth berbeda, periksa implementasi"
comparison.to_csv(RESULTS / "task6_comparison.csv", index=False)
pd.DataFrame(checks)

# Bagian E - Temporal Association Analysis
Memakai transaksi Author-Year yang dibagi menjadi empat periode. Support dihitung dengan penyebut jumlah transaksi masing-masing periode, dan ambang support serta confidence dibuat sama untuk semua periode.

## 1. Frequent pattern mining pada setiap periode

### Pembagian periode
Pembagian periode dan jumlah transaksi pada setiap periode. Periode 2024-2025 hanya dua tahun, jadi jumlah transaksinya lebih kecil.

In [ ]:
PERIODS = {"2015-2017": (2015, 2017), "2018-2020": (2018, 2020),
           "2021-2023": (2021, 2023), "2024-2025": (2024, 2025)}
temporal = {}


def run_period(period):
    start, end = PERIODS[period]
    subset = author_year_tx.loc[author_year_tx.year.between(start, end)]
    db = encode_transactions(subset["items"])
    found, seconds = mine(db, "FP-Growth", TEMPORAL_SUPPORT)
    rules = association_rules(found, db.n_transactions)
    temporal[period] = {"transactions": db.n_transactions, "itemsets": found, "rules": rules, "seconds": seconds}
    join_items(found, ["itemset"]).to_csv(RESULTS / f"temporal_itemsets_{start}_{end}.csv", index=False)
    join_items(rules, ["antecedent", "consequent"]).to_csv(RESULTS / f"temporal_rules_{start}_{end}.csv", index=False)
    print(f"{period}: {db.n_transactions:,} transaksi, {len(found):,} itemsets, {len(rules):,} rules, {seconds:.2f} detik")


pd.Series({period: int(author_year_tx.year.between(*span).sum()) for period, span in PERIODS.items()}).to_frame("transaksi")

### 2015-2017

In [ ]:
run_period("2015-2017")

### 2018-2020

In [ ]:
run_period("2018-2020")

### 2021-2023

In [ ]:
run_period("2021-2023")

### 2024-2025

In [ ]:
run_period("2024-2025")

## 2. Frequent itemsets pada setiap periode
Sepuluh frequent itemset dengan support tertinggi pada setiap periode.

In [ ]:
def show_period_itemsets(period, n=10):
    return join_items(temporal[period]["itemsets"].head(n).reset_index(drop=True), ["itemset"])

### 2015-2017

In [ ]:
show_period_itemsets("2015-2017")

### 2018-2020

In [ ]:
show_period_itemsets("2018-2020")

### 2021-2023

In [ ]:
show_period_itemsets("2021-2023")

### 2024-2025

In [ ]:
show_period_itemsets("2024-2025")

## 3. Association rules pada setiap periode
Sepuluh rule dengan lift tertinggi pada setiap periode.

In [ ]:
def show_period_rules(period, n=10):
    rules = temporal[period]["rules"]
    if rules.empty:
        return f"Tidak ada rule pada periode {period}"
    return format_rules(rules.head(n))

### 2015-2017

In [ ]:
show_period_rules("2015-2017")

### 2018-2020

In [ ]:
show_period_rules("2018-2020")

### 2021-2023

In [ ]:
show_period_rules("2021-2023")

### 2024-2025

In [ ]:
show_period_rules("2024-2025")

## 4. Perbandingan association rules antarperiode
Jumlah transaksi, frequent itemset, dan rule pada setiap periode dengan ambang yang sama.

In [ ]:
temporal_summary = pd.DataFrame([
    {"period": period, "transactions": t["transactions"], "itemsets": len(t["itemsets"]),
     "2-itemsets": int((t["itemsets"]["length"] == 2).sum()), "3+-itemsets": int((t["itemsets"]["length"] >= 3).sum()),
     "rules": len(t["rules"]), "seconds": round(t["seconds"], 2)}
    for period, t in temporal.items()
])
temporal_summary.to_csv(RESULTS / "temporal_summary.csv", index=False)
temporal_summary

## 5. Pola asosiasi yang bertahan dan yang berubah

### Kombinasi venue yang bertahan
Itemset berisi dua venue atau lebih yang frequent pada semua periode.

In [ ]:
itemset_sets = {period: {frozenset(s) for s in t["itemsets"].loc[t["itemsets"]["length"] >= 2, "itemset"]} for period, t in temporal.items()}
persistent_itemsets = set.intersection(*itemset_sets.values())
print("Kombinasi venue yang frequent di semua periode:", len(persistent_itemsets))
for combo in sorted(persistent_itemsets, key=lambda c: sorted(c))[:20]:
    print(" | ".join(sorted(combo)))

### Rule yang bertahan
Rule yang muncul di semua periode.

In [ ]:
rule_sets = {period: {(frozenset(a), frozenset(c)) for a, c in zip(t["rules"].antecedent, t["rules"].consequent)}
             for period, t in temporal.items()}
persistent = set.intersection(*rule_sets.values())
print("Rule yang muncul di semua periode:", len(persistent))
for a, c in sorted(persistent, key=lambda pair: (sorted(pair[0]), sorted(pair[1])))[:20]:
    print(" | ".join(sorted(a)), "->", " | ".join(sorted(c)))

### Rule yang berubah
Jumlah rule yang hanya muncul pada satu periode.

In [ ]:
pd.Series({
    period: len(ruleset - set.union(*(v for k, v in rule_sets.items() if k != period)))
    for period, ruleset in rule_sets.items()
}).to_frame("rule khusus periode")

### Metrik rule yang bertahan
Membandingkan support, confidence, dan lift dari rule yang muncul di semua periode.

In [ ]:
comparison_rows = []
for period, t in temporal.items():
    for _, row in t["rules"].iterrows():
        if (frozenset(row.antecedent), frozenset(row.consequent)) in persistent:
            comparison_rows.append({
                "period": period,
                "antecedent": " | ".join(sorted(row.antecedent)),
                "consequent": " | ".join(sorted(row.consequent)),
                "support": row.support, "confidence": row.confidence, "lift": row.lift,
            })
persistent_metrics = pd.DataFrame(comparison_rows)
if persistent_metrics.empty:
    print("Tidak ada rule yang bertahan di semua periode pada ambang ini.")
else:
    persistent_metrics.to_csv(RESULTS / "temporal_persistent_rules.csv", index=False)
    display(persistent_metrics.sort_values(["antecedent", "consequent", "period"]).head(40))

# Metadata Eksperimen
Menyimpan konfigurasi, tanggal snapshot, dan versi library run ini ke `results/main/run_metadata.json` agar eksperimen bisa direproduksi.

In [ ]:
metadata = {
    "snapshot_url": snapshot["source"], "snapshot_date": SNAPSHOT_DATE, "retrieved_at": snapshot["retrieved_at"],
    "run_utc": datetime.now(timezone.utc).isoformat(),
    "years": [YEAR_MIN, YEAR_MAX], "support_thresholds": SUPPORTS,
    "minimum_confidence": MIN_CONFIDENCE, "temporal_support": TEMPORAL_SUPPORT,
    "runtime_repeats": RUNTIME_REPEATS, "venue_aliases": VENUE_ALIASES,
    "algorithms": "Apriori, FP-Growth, dan association rules diimplementasikan sendiri di notebook",
    "python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
}
(RESULTS / "run_metadata.json").write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
con.close()
print("Metadata disimpan di", RESULTS / "run_metadata.json")